# Day 3 — Code Generator: Python to High-Performance Java

**Goal:** Use frontier LLMs to translate Python code into Java, and compare the generated code from different models.

In this notebook, we will:
1. Check API keys and connect to model providers.
2. Build a reusable prompt that asks a model to translate Python into Java.
3. Ask two different models to translate our Python benchmark.
4. Compare the generated Java code side-by-side.

## Step 1 — Import libraries

These libraries load environment variables, call model APIs, run compiler commands, and display formatted output.

In [2]:
import os
import subprocess #

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

## Step 2 — Load and check API keys

`load_dotenv(override=True)` reads values from your `.env` file. This cell only checks whether each key exists; it does not print the full secret key.

In [1]:
from google.colab import userdata


GROQ_API_KEY = userdata.get('GROQ_API_KEY')
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

api_keys = {
    "Google": GOOGLE_API_KEY,
    "Grok": GROQ_API_KEY,
}

for provider, api_key in api_keys.items():
    status = "available" if api_key else "not set (optional unless using this provider)"
    print(f"{provider}: {status}")

Google: available
Grok: available


## Step 3 — Connect to model providers

The OpenAI-compatible client interface lets us call multiple providers using a similar API. Each provider uses its own base URL and API key.

In [20]:
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq_url = "https://api.groq.com/openai/v1"

gemini = OpenAI(api_key=GOOGLE_API_KEY, base_url=gemini_url)
groq = OpenAI(api_key=GROQ_API_KEY, base_url=groq_url)

## Step 4 — Choose the models

In [24]:
OPENAI_MODEL = "openai/gpt-oss-120b"
GEMINI_MODEL = "gemini-3.6-flash"
GROQ_MODEL = "qwen/qwen3.8-27b"

## Step 5 — Create the Python-to-Java prompt

In [34]:
system_prompt = """
Your task is to convert Python code into high-performance Java code.
Respond only with valid Java code. Do not provide markdown formatting or explanations, except for occasional comments.
"""

In [35]:
def messages_for(python_code):
    return [
        {
            "role": "system", "content": system_prompt},
        {
            "role": "user", "content": f"Convert this Python code to Java:\n{python_code}"
        },
    ]

## Step 6 — Define the benchmark in Python

In [36]:
pi_benchmark_code = """
def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations + 1):
        j = i * param1 - param2
        result -= 1 / j
        j = i * param1 + param2
        result += 1 / j
    return result

result = calculate(200_000_000, 4, 1) * 4
print(f"Result: {result:.12f}")
"""

## Step 7 — Ask models to translate Python into Java

We define a helper function to call a model, request the translation, and return the generated code.

In [37]:
def port(client, model, python_code):
    response = client.chat.completions.create(
        model=model,
        messages=messages_for(python_code)
    )

    java_code = response.choices[0].message.content or ""

    # Strip markdown block formatting if returned by the model
    java_code = java_code.replace("```java", "").replace("```", "").strip()

    return java_code

## Step 8 — Translate Python code using each model

In [26]:
print(f"Translating with {OPENAI_MODEL}...")
openai_java = port(groq, OPENAI_MODEL, pi_benchmark_code)

print(f"Translating with {GEMINI_MODEL}...")
gemini_java = port(gemini, GEMINI_MODEL, pi_benchmark_code)

print(f"Translating with {GROQ_MODEL}...")
groq_java = port(groq, GROQ_MODEL, pi_benchmark_code)

Translating with openai/gpt-oss-120b...
Translating with gemini-3.6-flash...
Translating with qwen/qwen3.8-27b...


# Step 9 - Display the generated code

In [38]:
display(Markdown(f"### OpenAI\n```java\n{openai_java}\n```"))
display(Markdown(f"### Gemini\n```java\n{gemini_java}\n```"))
display(Markdown(f"### Groq\n```java\n{groq_java}\n```"))

### OpenAI
```java
public class Main {
    private static double calculate(int iterations, double param1, double param2) {
        double result = 1.0;
        for (int i = 1; i <= iterations; i++) {
            double j = i * param1 - param2;
            result -= 1.0 / j;
            j = i * param1 + param2;
            result += 1.0 / j;
        }
        return result;
    }

    public static void main(String[] args) {
        double result = calculate(200_000_000, 4.0, 1.0) * 4.0;
        System.out.printf("Result: %.12f%n", result);
    }
}
```

### Gemini
```java
import java.util.Locale;

public class Main {
    public static double calculate(long iterations, double param1, double param2) {
        double result = 1.0;
        for (long i = 1; i <= iterations; i++) {
            double j1 = i * param1 - param2;
            result -= 1.0 / j1;
            double j2 = i * param1 + param2;
            result += 1.0 / j2;
        }
        return result;
    }

    public static void main(String[] args) {
        double result = calculate(200_000_000, 4, 1) * 4;
        System.out.printf(Locale.US, "Result: %.12f%n", result);
    }
}
```

### Groq
```java
public class Main {
    public static void main(String[] args) {
        int iterations = 200_000_000;
        int param1 = 4;
        int param2 = 1;

        double result = calculate(iterations, param1, param2) * 4;
        System.out.printf("Result: %.12f%n", result);
    }

    private static double calculate(int iterations, int param1, int param2) {
        double result = 1.0;
        for (int i = 1; i <= iterations; i++) {
            double j = (double) i * param1 - param2;
            result -= 1.0 / j;
            j = (double) i * param1 + param2;
            result += 1.0 / j;
        }
        return result;
    }
}
```

# Step 10 - Execution Time Comparison

Let's compile each generated Java code snippet, execute them multiple times, and measure their actual execution speeds side-by-side.

In [39]:
import time
import subprocess

# === 1. Save the Java files ===
with open("MainOpenAI.java", "w") as f:
    f.write(openai_java.replace("class Main", "class MainOpenAI"))

with open("MainGemini.java", "w") as f:
    f.write(gemini_java.replace("class Main", "class MainGemini"))

with open("MainGroq.java", "w") as f:
    f.write(groq_java.replace("class Main", "class MainGroq"))

# === 2. Compile each file ===
print("Compiling Java files...")
subprocess.run(["javac", "MainOpenAI.java"])
subprocess.run(["javac", "MainGemini.java"])
subprocess.run(["javac", "MainGroq.java"])

# === 3. Measure Execution Time ===
print("\nMeasuring execution times...")

# Time OpenAI Code
start = time.time()
subprocess.run(["java", "MainOpenAI"], capture_output=True)
openai_time = time.time() - start
print(f"OpenAI Model: {openai_time:.4f} seconds")

# Time Gemini Code
start = time.time()
subprocess.run(["java", "MainGemini"], capture_output=True)
gemini_time = time.time() - start
print(f"Gemini Model: {gemini_time:.4f} seconds")

# Time Groq Code
start = time.time()
subprocess.run(["java", "MainGroq"], capture_output=True)
groq_time = time.time() - start
print(f"Groq Model: {groq_time:.4f} seconds")

Compiling Java files...

Measuring execution times...
OpenAI Model: 2.7026 seconds
Gemini Model: 2.9424 seconds
Groq Model: 3.1625 seconds
